*Add a privacy mechanism to your project so that client model updates are clipped and noise is added before they are shared for aggregation.
- Gradient/update clipping: Limits how much a model update can change.
- Gaussian noise: Adds controlled random noise to the update.
- Privacy–utility trade-off: More noise can improve privacy but reduce model accuracy.
This is a prototype privacy mechanism, not yet a formally verified differential-privacy guarantee. We will not claim a specific privacy budget (\(\varepsilon,\delta\)) without proper accounting.*

In [14]:
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

BASE = Path("D:/adaptive_federated_gnn")
GRAPH_PATH = BASE / "data/processed/edge_graph.pt"
DAY10_PATH = BASE / "outputs/day10/federated_clients.pt"
DAY16_PATH = BASE / "outputs/day16"
OUT = BASE / "outputs/day17"
OUT.mkdir(parents=True, exist_ok=True)

torch.manual_seed(42)
np.random.seed(42)

graph = torch.load(GRAPH_PATH, map_location="cpu", weights_only=False)
day10_clients = torch.load(
    DAY10_PATH, map_location="cpu", weights_only=False
)
clients = torch.load(
    DAY16_PATH / "non_iid_clients.pt",
    map_location="cpu", weights_only=False
)

edge_index = graph["edge_index"].long()
feature_key = "edge_attr" if "edge_attr" in graph else "edge_features"
label_key = "edge_label" if "edge_label" in graph else "edge_labels"

edge_features = graph[feature_key].float()
edge_labels = graph[label_key].long().view(-1)

num_nodes = int(edge_index.max().item()) + 1
feature_size = edge_features.shape[1]

local_models = {}
for name in ["client_1", "client_2", "client_3"]:
    model_path = DAY16_PATH / f"{name}_local_model.pt"
    model_state = torch.load(
        model_path, map_location="cpu", weights_only=True
    )
    local_models[name] = model_state

print("Graph loaded:", tuple(edge_features.shape))
print("Clients loaded:", list(local_models.keys()))

Graph loaded: (286095, 67)
Clients loaded: ['client_1', 'client_2', 'client_3']


In [15]:
class EdgeGCN(nn.Module):
    def __init__(self, num_nodes, edge_feature_size,
                 embedding_size=16, hidden_size=32):
        super().__init__()
        self.node_embeddings = nn.Embedding(num_nodes, embedding_size)
        self.conv1 = GCNConv(embedding_size, hidden_size)
        self.conv2 = GCNConv(hidden_size, hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2 + edge_feature_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, edge_index, edge_features):
        node_ids = torch.arange(
            self.node_embeddings.num_embeddings,
            device=edge_index.device
        )
        x = self.node_embeddings(node_ids)
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        source = x[edge_index[0]]
        destination = x[edge_index[1]]
        combined = torch.cat(
            [source, destination, edge_features], dim=1
        )
        return self.classifier(combined)

print("EdgeGCN architecture defined.")

EdgeGCN architecture defined.


In [16]:
torch.manual_seed(42)

initial_model = EdgeGCN(num_nodes, feature_size)
initial_state = {
    key: value.detach().clone()
    for key, value in initial_model.state_dict().items()
}

# Verify that the recreated starting state matches Day 16's
# training initialization before using it.
def states_equal(a, b):
    return all(
        torch.equal(a[k], b[k])
        for k in a
    )

reference_model = EdgeGCN(num_nodes, feature_size)
reference_state = reference_model.state_dict()

print("Starting model created.")
print("Parameter tensors:", len(initial_state))

Starting model created.
Parameter tensors: 9


In [17]:
INITIAL_PATH = OUT / "shared_initial_model.pt"

torch.manual_seed(1701)
shared_model = EdgeGCN(num_nodes, feature_size)
shared_state = {
    key: value.detach().clone()
    for key, value in shared_model.state_dict().items()
}
torch.save(shared_state, INITIAL_PATH)

retrained_states = {}

for name, data in clients.items():
    model = EdgeGCN(num_nodes, feature_size)
    model.load_state_dict(copy.deepcopy(shared_state))

    idx = data["local_edge_indices"].long()
    local_edge_index = edge_index[:, idx]
    x_local = data["edge_features"].float()
    y_local = data["edge_labels"].long()

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(2):
        optimizer.zero_grad()
        logits = model(local_edge_index, x_local)
        loss = loss_fn(logits, y_local)
        loss.backward()
        optimizer.step()

    state = {
        key: value.detach().clone()
        for key, value in model.state_dict().items()
    }
    retrained_states[name] = state
    torch.save(state, OUT / f"{name}_retrained_model.pt")

    print(f"{name}: final loss = {loss.item():.4f}")

print("All clients trained from the same saved starting model.")

client_1: final loss = 0.6906
client_2: final loss = 0.7146
client_3: final loss = 0.6398
All clients trained from the same saved starting model.


In [18]:
CLIP_NORM = 1.0
NOISE_MULTIPLIER = 0.5

def protect_client_update(local_state, base_state,
                          clip_norm=1.0, noise_multiplier=0.5):
    # Compute one global L2 norm over all parameter updates.
    deltas = {
        key: local_state[key].float() - base_state[key].float()
        for key in base_state
    }

    squared_norm = sum(
        torch.sum(delta ** 2) for delta in deltas.values()
    )
    update_norm = torch.sqrt(squared_norm)

    scale = min(
        1.0,
        clip_norm / (update_norm.item() + 1e-12)
    )

    protected = {}
    for key, delta in deltas.items():
        clipped_delta = delta * scale
        noise = torch.randn_like(clipped_delta) * (
            noise_multiplier * clip_norm
        )
        protected[key] = clipped_delta + noise

    return protected, update_norm.item(), scale

protected_updates = {}
privacy_rows = []

for name, state in retrained_states.items():
    update, original_norm, clip_scale = protect_client_update(
        state,
        shared_state,
        CLIP_NORM,
        NOISE_MULTIPLIER
    )

    protected_updates[name] = update

    privacy_rows.append({
        "Client": name,
        "Original Update Norm": original_norm,
        "Clip Scale": clip_scale,
        "Clip Norm": CLIP_NORM,
        "Noise Multiplier": NOISE_MULTIPLIER
    })

privacy_df = pd.DataFrame(privacy_rows)
display(privacy_df)

privacy_df.to_csv(OUT / "privacy_update_diagnostics.csv", index=False)
torch.save(protected_updates, OUT / "protected_client_updates.pt")

print("Protected client updates saved.")

,Client,Original Update Norm,Clip Scale,Clip Norm,Noise Multiplier
0,client_1,0.400369,1.0,1.0,0.5
1,client_2,0.305792,1.0,1.0,0.5
2,client_3,0.465926,1.0,1.0,0.5


Protected client updates saved.


In [19]:
from pathlib import Path
import copy
import torch
import pandas as pd

OUT = Path("D:/adaptive_federated_gnn/outputs/day17")
OUT.mkdir(parents=True, exist_ok=True)

# Verify required variables from earlier cells exist.
required = [
    "shared_state", "protected_updates",
    "clients", "num_nodes", "feature_size"
]
missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. Run Cells 1–5 first."
    )

client_sizes = {
    name: len(data["edge_labels"])
    for name, data in clients.items()
}
total_size = sum(client_sizes.values())

weights = {
    name: size / total_size
    for name, size in client_sizes.items()
}

private_global_state = {}

for key, base_tensor in shared_state.items():
    combined_delta = torch.zeros_like(
        base_tensor, dtype=torch.float32
    )

    for name in protected_updates:
        combined_delta += (
            weights[name] * protected_updates[name][key].float()
        )

    private_global_state[key] = (
        base_tensor.float() + combined_delta
    )

private_model = EdgeGCN(num_nodes, feature_size)
private_model.load_state_dict(private_global_state)

model_path = OUT / "private_global_model.pt"
torch.save(private_model.state_dict(), model_path)

pd.DataFrame([
    {
        "Client": name,
        "Training Edges": client_sizes[name],
        "Aggregation Weight": weights[name]
    }
    for name in weights
]).to_csv(OUT / "private_fedavg_weights.csv", index=False)

print("Global model saved:", model_path)
print("Model file exists:", model_path.exists())

Global model saved: D:\adaptive_federated_gnn\outputs\day17\private_global_model.pt
Model file exists: True


In [20]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

OUT = Path("D:/adaptive_federated_gnn/outputs/day17")
DAY10_PATH = Path(
    "D:/adaptive_federated_gnn/outputs/day10/federated_clients.pt"
)

# Load the original client split to preserve the held-out test set.
day10_clients = torch.load(
    DAY10_PATH, map_location="cpu", weights_only=False
)

train_indices = np.unique(np.concatenate([
    data["local_edge_indices"].cpu().numpy()
    if torch.is_tensor(data["local_edge_indices"])
    else np.asarray(data["local_edge_indices"])
    for data in day10_clients.values()
])).astype(np.int64)

train_mask = np.zeros(len(edge_labels), dtype=bool)
train_mask[train_indices] = True
test_indices = np.flatnonzero(~train_mask)

if len(test_indices) == 0:
    raise RuntimeError("No held-out test edges found.")

test_idx = torch.tensor(test_indices, dtype=torch.long)
test_y = edge_labels[test_idx].cpu().numpy()

# Load the saved model to verify it can be restored.
private_model = EdgeGCN(num_nodes, feature_size)
private_model.load_state_dict(torch.load(
    OUT / "private_global_model.pt",
    map_location="cpu",
    weights_only=True
))
private_model.eval()

with torch.no_grad():
    logits = private_model(edge_index, edge_features)[test_idx]
    predictions = logits.argmax(dim=1).cpu().numpy()

metrics = {
    "Accuracy": accuracy_score(test_y, predictions),
    "Precision": precision_score(
        test_y, predictions, zero_division=0
    ),
    "Recall": recall_score(
        test_y, predictions, zero_division=0
    ),
    "F1 Score": f1_score(
        test_y, predictions, zero_division=0
    )
}

metrics_path = OUT / "private_model_test_metrics.csv"
pd.DataFrame([metrics]).to_csv(metrics_path, index=False)

predictions_path = OUT / "private_model_test_predictions.csv"
pd.DataFrame({
    "Edge Index": test_indices,
    "Actual Label": test_y,
    "Predicted Label": predictions
}).to_csv(predictions_path, index=False)

print("Evaluation completed.")
print("Metrics saved:", metrics_path)
print("Metrics file exists:", metrics_path.exists())
display(pd.DataFrame([metrics]))

Evaluation completed.
Metrics saved: D:\adaptive_federated_gnn\outputs\day17\private_model_test_metrics.csv
Metrics file exists: True


,Accuracy,Precision,Recall,F1 Score
0,0.57279,0.565121,0.999475,0.722007


In [21]:
from pathlib import Path
import pandas as pd

OUT = Path("D:/adaptive_federated_gnn/outputs/day17")

required_files = [
    "private_global_model.pt",
    "private_fedavg_weights.csv",
    "private_model_test_metrics.csv",
    "private_model_test_predictions.csv"
]

print("DAY 17 OUTPUT CHECK")
for filename in required_files:
    path = OUT / filename
    print(f"{filename}: {'OK' if path.exists() else 'MISSING'}")

metrics_path = OUT / "private_model_test_metrics.csv"

if metrics_path.exists():
    print("\nPrivate model test metrics:")
    display(pd.read_csv(metrics_path))
else:
    print(
        "\nMetrics file is still missing. "
        "Check whether Cell 7 completed successfully."
    )

DAY 17 OUTPUT CHECK
private_global_model.pt: OK
private_fedavg_weights.csv: OK
private_model_test_metrics.csv: OK
private_model_test_predictions.csv: OK

Private model test metrics:


,Accuracy,Precision,Recall,F1 Score
0,0.57279,0.565121,0.999475,0.722007
